In [1]:
import pandas as pd

LYCOS17 = r"D:\thesis_implementation\LycoS-IDS2017_CLEANED.csv"
LYCOS18 = r"D:\thesis_implementation\LycoS-Unicas-IDS2018_CLEANED.csv"

for name, path in [("LycoS17", LYCOS17), ("LycoS18", LYCOS18)]:
    df = pd.read_csv(path, nrows=500000)
    print(f"\n{'='*50}")
    print(f"  Dataset : {name}")
    print(f"  Columns : {len(df.columns)}")
    print(f"  NaN     : {df.isnull().sum().sum()}")
    print(f"  Inf     : {(df.select_dtypes('number') == float('inf')).sum().sum()}")
    print(f"\n  Label column name : '{df.columns[-1]}'")
    print(f"  Unique labels     :\n{df.iloc[:, -1].value_counts()}")


  Dataset : LycoS17
  Columns : 83
  NaN     : 0
  Inf     : 0

  Label column name : 'label'
  Unique labels     :
label
0    262166
1    237834
Name: count, dtype: int64

  Dataset : LycoS18
  Columns : 78
  NaN     : 0
  Inf     : 0

  Label column name : 'label'
  Unique labels     :
label
0    500000
Name: count, dtype: int64


In [ ]:
#02_diagnose the cloumns numbers 
import pandas as pd

LYCOS17 = r"D:\thesis_implementation\LycoS-IDS2017_CLEANED.csv"
LYCOS18 = r"D:\thesis_implementation\LycoS-Unicas-IDS2018_CLEANED.csv"

# --- LycoS17: find the 5 extra columns ---
print("===== LycoS17 ALL COLUMN NAMES =====")
df17 = pd.read_csv(LYCOS17, nrows=5)
for i, col in enumerate(df17.columns):
    print(f"  [{i}] {col}")

# --- LycoS18: full label distribution (chunked to handle 13M rows) ---
print("\n===== LycoS18 FULL LABEL DISTRIBUTION =====")
label_counts = {}
for chunk in pd.read_csv(LYCOS18, chunksize=500000, usecols=['label']):
    for val, cnt in chunk['label'].value_counts().items():
        label_counts[val] = label_counts.get(val, 0) + cnt

for label, count in sorted(label_counts.items()):
    print(f"  {label}: {count:,}")
print(f"  TOTAL: {sum(label_counts.values()):,}")

===== LycoS17 ALL COLUMN NAMES =====
  [0] flow_id
  [1] src_addr
  [2] src_port
  [3] dst_addr
  [4] dst_port
  [5] ip_prot
  [6] timestamp
  [7] flow_duration
  [8] down_up_ratio
  [9] pkt_len_max
  [10] pkt_len_min
  [11] pkt_len_mean
  [12] pkt_len_var
  [13] pkt_len_std
  [14] bytes_per_s
  [15] pkt_per_s
  [16] fwd_pkt_per_s
  [17] bwd_pkt_per_s
  [18] fwd_pkt_cnt
  [19] fwd_pkt_len_tot
  [20] fwd_pkt_len_max
  [21] fwd_pkt_len_min
  [22] fwd_pkt_len_mean
  [23] fwd_pkt_len_std
  [24] fwd_pkt_hdr_len_tot
  [25] fwd_pkt_hdr_len_min
  [26] fwd_non_empty_pkt_cnt
  [27] bwd_pkt_cnt
  [28] bwd_pkt_len_tot
  [29] bwd_pkt_len_max
  [30] bwd_pkt_len_min
  [31] bwd_pkt_len_mean
  [32] bwd_pkt_len_std
  [33] bwd_pkt_hdr_len_tot
  [34] bwd_pkt_hdr_len_min
  [35] bwd_non_empty_pkt_cnt
  [36] iat_max
  [37] iat_min
  [38] iat_mean
  [39] iat_std
  [40] fwd_iat_tot
  [41] fwd_iat_max
  [42] fwd_iat_min
  [43] fwd_iat_mean
  [44] fwd_iat_std
  [45] bwd_iat_tot
  [46] bwd_iat_max
  [47] bwd_iat_

In [ ]:
#03_fix_lycos17 and dropping features that need to be removed, Do not run this code more than once
import pandas as pd

LYCOS17_IN  = r"D:\thesis_implementation\LycoS-IDS2017_CLEANED.csv"
LYCOS17_OUT = r"D:\thesis_implementation\LycoS-IDS2017_FINAL.csv"

COLS_TO_DROP = ['flow_id', 'src_addr', 'dst_addr', 'src_port', 'timestamp']

print("Loading LycoS17...")
df = pd.read_csv(LYCOS17_IN)
print(f"  Shape before: {df.shape}")

df = df.drop(columns=COLS_TO_DROP)
print(f"  Shape after : {df.shape}")

# Verify columns now match LycoS18 (77 features + label)
assert df.shape[1] == 78, f"Expected 78 columns, got {df.shape[1]}"
assert 'label' in df.columns, "label column missing!"
assert df.isnull().sum().sum() == 0, "NaN values found!"

print(f"  Label distribution:\n{df['label'].value_counts()}")
print("\nSaving FINAL file...")
df.to_csv(LYCOS17_OUT, index=False)
print(f"  Saved to: {LYCOS17_OUT}")
print("\n✅ LycoS17 FINAL is ready!")

Loading LycoS17...
  Shape before: (1837498, 83)
  Shape after : (1837498, 78)
  Label distribution:
label
0    1395675
1     441823
Name: count, dtype: int64

Saving FINAL file...
  Saved to: D:\thesis_implementation\LycoS-IDS2017_FINAL.csv

✅ LycoS17 FINAL is ready!


In [4]:
#04_verify_final_checks 
import pandas as pd

LYCOS17 = r"D:\thesis_implementation\LycoS-IDS2017_FINAL.csv"
LYCOS18 = r"D:\thesis_implementation\LycoS-Unicas-IDS2018_CLEANED.csv"

datasets = [("LycoS17", LYCOS17), ("LycoS18", LYCOS18)]

for name, path in datasets:
    print(f"\n{'='*55}")
    print(f"  DATASET : {name}")
    print(f"{'='*55}")

    df = pd.read_csv(path, nrows=500000)
    full_counts = {}
    for chunk in pd.read_csv(path, chunksize=500000, usecols=['label']):
        for val, cnt in chunk['label'].value_counts().items():
            full_counts[val] = full_counts.get(val, 0) + cnt

    total = sum(full_counts.values())

    print(f"  Columns          : {len(df.columns)}")
    print(f"  Feature columns  : {len(df.columns) - 1}")
    print(f"  Label column     : '{df.columns[-1]}'")
    print(f"  Total rows       : {total:,}")
    print(f"\n  Label distribution:")
    for label, count in sorted(full_counts.items()):
        pct = count / total * 100
        tag = "Benign" if label == 0 else "Attack"
        print(f"    {label} ({tag}): {count:,}  ({pct:.2f}%)")

    print(f"\n  NaN values       : {df.isnull().sum().sum()}")
    print(f"  Inf values       : {(df.select_dtypes('number') == float('inf')).sum().sum()}")
    print(f"  Dtypes:")
    print(f"    Numeric cols   : {len(df.select_dtypes('number').columns)}")
    print(f"    Non-numeric    : {list(df.select_dtypes(exclude='number').columns)}")

    print(f"\n  Feature names match between datasets check...")

# Column alignment check
df17 = pd.read_csv(LYCOS17, nrows=1)
df18 = pd.read_csv(LYCOS18, nrows=1)
cols17 = set(df17.columns)
cols18 = set(df18.columns)

print(f"\n{'='*55}")
print(f"  CROSS-DATASET COLUMN ALIGNMENT")
print(f"{'='*55}")
print(f"  Columns in LycoS17 only : {cols17 - cols18}")
print(f"  Columns in LycoS18 only : {cols18 - cols17}")
print(f"  Shared columns          : {len(cols17 & cols18)}")

if cols17 == cols18:
    print("\n  ✅ Both datasets have IDENTICAL columns — ready for cross-dataset training!")
else:
    print("\n  ⚠️  Column mismatch — needs fixing before training!")


  DATASET : LycoS17
  Columns          : 78
  Feature columns  : 77
  Label column     : 'label'
  Total rows       : 1,837,498

  Label distribution:
    0 (Benign): 1,395,675  (75.96%)
    1 (Attack): 441,823  (24.04%)

  NaN values       : 0
  Inf values       : 0
  Dtypes:
    Numeric cols   : 78
    Non-numeric    : []

  Feature names match between datasets check...

  DATASET : LycoS18
  Columns          : 78
  Feature columns  : 77
  Label column     : 'label'
  Total rows       : 13,691,268

  Label distribution:
    0 (Benign): 10,000,000  (73.04%)
    1 (Attack): 3,691,268  (26.96%)

  NaN values       : 0
  Inf values       : 0
  Dtypes:
    Numeric cols   : 78
    Non-numeric    : []

  Feature names match between datasets check...

  CROSS-DATASET COLUMN ALIGNMENT
  Columns in LycoS17 only : set()
  Columns in LycoS18 only : set()
  Shared columns          : 78

  ✅ Both datasets have IDENTICAL columns — ready for cross-dataset training!
